In [1]:
import os, re
import numpy as np
import scanpy as sc
from os.path import join
import pandas as pd

import sys
import scipy.io as sio
import scipy.sparse as sps
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from spamosaic.framework import SpaMosaic
import spamosaic.utils as utls
from spamosaic.preprocessing import RNA_preprocess, ADT_preprocess, Epigenome_preprocess, harmony

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8' 

In [2]:
data_dir = '../../../data/processed/Misar-E15-E18-E18'

ad_mult_rna = sc.read_h5ad(join(data_dir, 'E15/ad_rna.h5ad'))
ad_mult_atac = sc.read_h5ad(join(data_dir, 'E15/ad_atac.h5ad'))

ad_test_rna = sc.read_h5ad(join(data_dir, 'E18-rna/ad_rna.h5ad'))
ad_test_atac = sc.read_h5ad(join(data_dir, 'E18-atac/ad_atac.h5ad'))

input_dict = { 
    'rna':   [ad_mult_rna, ad_test_rna, None],
    'atac':  [ad_mult_atac,None,        ad_test_atac],
}

input_key = 'dimred_bc'
batch_key = 'Sample'

In [3]:
Epigenome_preprocess(input_dict['atac'], batch_corr=True, n_peak=50000, batch_key=batch_key, key=input_key, return_hvf=False)
RNA_preprocess(input_dict['rna'], batch_corr=True, n_hvg=10000, batch_key=batch_key, key=input_key)

Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
	Completed 8 / 10 iteration(s).
	Completed 9 / 10 iteration(s).
	Completed 10 / 10 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
Reach convergence after 3 iteration(s).


In [4]:
model = SpaMosaic(
    modBatch_dict=input_dict, input_key=input_key,
    batch_key=batch_key, 
    intra_knns=10, inter_knn_base=10, w_g=0.8,
    seed=1234, 
    device='cuda:0'
)

model.train(net='wlgcn', lr=0.01, T=0.01, n_epochs=100)

batch0: ['rna', 'atac']
batch1: ['rna']
batch2: ['atac']
------Calculating spatial graph...
The graph contains 19490 edges, 1949 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 21290 edges, 2129 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 19490 edges, 1949 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 21290 edges, 2129 cells.
10.0000 neighbors per cell on average.
Searching MNN within rna
Finding MNN between (E15_5-S1, S2) using KNN (10, 10)
Number of mnn pairs for rna:2026
Searching MNN within atac
Finding MNN between (E15_5-S1, S2) using KNN (10, 10)
Number of mnn pairs for atac:8089


100%|███████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 44.57it/s]


In [5]:
ad_embs = model.infer_emb(input_dict, emb_key='emb', final_latent_key='merged_emb')
ad_mosaic = sc.concat(ad_embs)
ad_mosaic = utls.get_umap(ad_mosaic, use_reps=['merged_emb'])

ad_mosaic = utls.clustering(ad_mosaic, 14, 'merged_emb', algo='mclust', key='mclust')
utls.split_adata_ob(ad_embs, ad_mosaic, ob='obs', key='mclust')

/home/yanxh/anaconda3/envs/spamosaic-env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
R[write to console]:                    __           __ 
   ____ ___  _____/ /_  _______/ /_
  / __ `__ \/ ___/ / / / / ___/ __/
 / / / / / / /__/ / /_/ (__  ) /_  
/_/ /_/ /_/\___/_/\__,_/____/\__/   version 6.0.0
Type 'citation("mclust")' for citing this R package in publications.



fitting ...
  |======================================================================| 100%


In [9]:
fig_dir = f'../../../results/embeddings/SpaMosaic/Misar-E15-E18-E18'
os.makedirs(fig_dir, exist_ok=True)

df = pd.DataFrame(ad_mosaic.obsm['merged_emb'], index=ad_mosaic.obs_names)
df.to_csv(join(fig_dir, 'df_emb.csv'))